**Mô tả bài tập: Phân loại đa lớp sử dụng mô hình CLIP**

Phân loại đa lớp (multi-label classification) là bài toán trong đó **mỗi ảnh có thể thuộc về nhiều lớp cùng lúc**, thay vì chỉ một lớp duy nhất như bài toán phân loại truyền thống (single-label classification). Điều này phản ánh đúng hơn các tình huống trong thực tế, ví dụ: một bức ảnh có thể đồng thời chứa _dog_, _chair_ và _sofa_.

Trong bài toán này, mục tiêu của mô hình là:

- Nhận một ảnh đầu vào.
- Dự đoán một **tập các nhãn** phù hợp với ảnh.

Khác với phân loại đơn lớp, mô hình phải xử lý mối quan hệ giữa nhiều nhãn cùng lúc, khiến bài toán trở nên phức tạp hơn nhưng cũng thực tế và hữu ích trong nhiều ứng dụng thị giác máy tính.

Trong bài tập này, chúng ta sẽ sử dụng mô hình **CLIP (Contrastive Language–Image Pre-training)** để giải quyết bài toán **phân loại đa lớp (multi-label classification)** trên tập dữ liệu hình ảnh. CLIP là mô hình được huấn luyện trước trên tập dữ liệu khổng lồ gồm hàng trăm triệu cặp văn bản – hình ảnh, giúp mô hình hiểu và liên kết được nội dung thị giác của ảnh với các mô tả ngôn ngữ tự nhiên.

**Mục tiêu của bài tập:**

- Sử dụng CLIP để trích xuất đặc trưng từ ảnh và từ các mô tả văn bản tương ứng với từng lớp (class text prompts).
- Tính toán độ tương đồng (similarity) giữa embedding ảnh và embedding văn bản.
- Xác định nhãn dự đoán cho ảnh bằng cách so sánh độ tương đồng với các lớp và chuyển kết quả thành dạng multi-label.
- Đánh giá khả năng phân loại đa lớp của CLIP trên tập dữ liệu đã cho.

Một điểm nổi bật của CLIP là mô hình hoạt động theo cơ chế **zero-shot**, tức là không cần huấn luyện lại. Chỉ cần cung cấp mô tả văn bản phù hợp cho từng lớp, mô hình có thể linh hoạt mở rộng sang nhiều nhiệm vụ phân loại khác nhau mà không cần fine-tuning.


**Import các thư viện cần thiết**


In [ ]:
import json
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    average_precision_score,
)
from transformers import CLIPProcessor, CLIPModel
import random
import matplotlib.pyplot as plt
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

**Load mô hình CLIP vào CPU để xử lý**

Trong đó _model_ dùng để suy luận còn _processor_ để xử lý dữ liệu


In [ ]:
device = "cpu"

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32",
    # local_files_only=True   # tạm comment dòng nếu dùng colab, dùng colab cần phải tải lại. Nhưng cần dòng này khi chạy trên hệ thống
).to("cpu")

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32",
    # local_files_only=True   # tạm comment dòng nếu dùng colab, dùng colab cần phải tải lại. Nhưng cần dòng này khi chạy trên hệ thống
)

**Tải dữ liệu test**


In [ ]:
!gdown 1EwaMW_EklBR3KzWkMJ2lX-X20JIunWoY
if os.path.exists("./VOC_CS431") == False:
    print("File not found!")
    !unzip -q VOC_CS431.zip

**Trực quan một số ví dụ**


In [ ]:
json_path = "./VOC_CS431/test.json"
with open(json_path, "r") as f:
    data = json.load(f)

IMAGE_DIR = "./VOC_CS431/VOC_CS431"

chosen_ids = ["000019", "000711", "000044", "000266", "000107", "000699"]

images = []
labels = []

for img_id in chosen_ids:
    img_path = os.path.join(IMAGE_DIR, f"{img_id}.jpg")

    if not os.path.exists(img_path):
        print(f"[WARNING] Không tìm thấy ảnh: {img_path}")
        continue

    # resize cho đồng nhất
    images.append(Image.open(img_path).resize((500, 500)))
    labels.append(data[img_id])  # ví dụ: "cat dog"

# ========================== Plot 6 ảnh ==========================
plt.figure(figsize=(15, 10))

for i, (img, lab) in enumerate(zip(images, labels)):
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(f"{chosen_ids[i]}: {lab}", fontsize=13)
    plt.axis("off")

plt.tight_layout()
plt.show()

**Thử nghiệm suy luận bằng mô hình CLIP với các lớp dự đoán** bao gồm _"dog", "cat", "potted plant", "chair", "sofa", "sheep", "cow"_


In [ ]:
with open("./VOC_CS431/test.json", "r") as f:
    data = json.load(f)


prompts = ["dog", "cat", "potted plant", "chair", "sofa", "sheep", "cow"]

image_ids = list(data.keys())
chosen_ids = ["000019", "000711", "000044", "000266", "000107", "000699"]

results = []

for img_id in chosen_ids:

    img_path = f"./VOC_CS431/VOC_CS431/{img_id}.jpg"
    image = Image.open(img_path).convert("RGB")
    img_inputs = processor(images=image, return_tensors="pt").to(device)
    text_inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        image_embed = model.get_image_features(**img_inputs)
        image_embed = image_embed / image_embed.norm()

        text_embeds = model.get_text_features(**text_inputs)
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

    similarity = (image_embed @ text_embeds.T).squeeze(0)
    probs = similarity

    pred_list = []
    for cls, p in zip(prompts, probs.tolist()):
        pred_list.append(f"{cls}: {p:.2f}")

    pred_text = "\n".join(pred_list)

    results.append((img_id, image, pred_text, data[img_id]))

In [ ]:
plt.figure(figsize=(18, 10))

for i, (img_id, img, pred_text, gt) in enumerate(results):
    plt.subplot(2, 3, i + 1)
    plt.imshow(img.resize((500, 500)))

    plt.title(
        f"ID: {img_id}\n\nGround truth: {gt}\n\nPredictions:\n{pred_text}",
        fontsize=11,
        loc="left",
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

### Yêu cầu phần mã trong `BEGIN SOLUTION` và `END SOLUTION`

Trong phần này, bạn cần viết đoạn mã thực hiện việc dự đoán nhãn đa lớp cho các ảnh đầu vào.

**Input:**

- `json`: dữ liệu chứa danh sách ID ảnh và nhãn tương ứng (nhãn chỉ xuất hiện trong bộ public test).
- `classes`: danh sách các lớp cần dự đoán.
- Các ảnh được lưu trong thư mục quy định, mỗi ảnh có tên dạng `<id>.jpg`.

**Output:**

- Một dictionary `y_pred_dict` có cấu trúc:
  - **Key:** ID của ảnh (không bao gồm `.jpg`).
  - **Value:** vector multi-hot biểu diễn các lớp được mô hình dự đoán (ví dụ: `[0, 1, 0, 0, 1, ...]`).
- Độ dài vector bằng đúng số lớp trong `classes`.

Nhiệm vụ của bạn là đọc ảnh, chạy mô hình dự đoán, và chuyển kết quả thu được thành vector multi-hot theo đúng định dạng yêu cầu.


In [33]:
def get_image_views(image, size=224):
    """Generates multiple views for TTA (Test Time Augmentation)."""
    # View 1: Squish (Resize directly) - Good for small details/textures
    squish = image.resize((size, size), Image.Resampling.BICUBIC)

    # View 2: Letterbox (Pad with black) - Good for shape/aspect ratio
    w, h = image.size
    scale = size / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
    letterbox = Image.new("RGB", (size, size), (0, 0, 0))
    letterbox.paste(resized, ((size - new_w) // 2, (size - new_h) // 2))

    # Return 4 views: Original + Flips
    return [
        squish,
        squish.transpose(Image.FLIP_LEFT_RIGHT),
        letterbox,
        letterbox.transpose(Image.FLIP_LEFT_RIGHT),
    ]


def run(json_data, classes):
    IMAGE_DIR = "./VOC_CS431/VOC_CS431"
    y_pred_dict = {}

    # Initialize results
    for img_id in json_data:
        y_pred_dict[img_id] = [0] * len(classes)

    ### BEGIN SOLUTION

    # --- 2. Clean Class Names ---
    class_map = {
        "pottedplant": "a potted plant",
        "diningtable": "a dining table",
        "tvmonitor": "a tv monitor",
        "motorbike": "a motorcycle",
        "aeroplane": "an airplane",
        "sofa": "a sofa",
        "bicycle": "a bicycle",
        "bird": "a bird",
        "boat": "a boat",
        "bottle": "a bottle",
        "bus": "a bus",
        "car": "a car",
        "cat": "a cat",
        "chair": "a chair",
        "cow": "a cow",
        "dog": "a dog",
        "horse": "a horse",
        "person": "a person",
        "sheep": "a sheep",
        "train": "a train",
    }
    cleaned_classes = [class_map.get(c, c) for c in classes]

    # --- 3. Prompt Ensembling (Standard 80 Templates) ---
    templates = [
        "a photo of a {}.",
        "a bad photo of a {}.",
        "a photo of many {}.",
        "a sculpture of a {}.",
        "a photo of the hard to see {}.",
        "a low resolution photo of the {}.",
        "a rendering of a {}.",
        "graffiti of a {}.",
        "a bad photo of the {}.",
        "a cropped photo of the {}.",
        "a tattoo of a {}.",
        "the embroidered {}.",
        "a photo of a hard to see {}.",
        "a bright photo of a {}.",
        "a photo of a clean {}.",
        "a photo of a dirty {}.",
        "a dark photo of the {}.",
        "a drawing of a {}.",
        "a photo of my {}.",
        "the plastic {}.",
        "a photo of the cool {}.",
        "a close-up photo of a {}.",
        "a black and white photo of the {}.",
        "a painting of the {}.",
        "a painting of a {}.",
        "a pixelated photo of the {}.",
        "a sculpture of the {}.",
        "a bright photo of the {}.",
        "a cropped photo of a {}.",
        "a plastic {}.",
        "a photo of the dirty {}.",
        "a jpeg photo of a {}.",
        "a blurry photo of the {}.",
        "a photo of the {}.",
        "a good photo of the {}.",
        "a rendering of the {}.",
        "a {} in a video game.",
        "a photo of one {}.",
        "a doodle of a {}.",
        "a close-up photo of the {}.",
        "the origami {}.",
        "the {} in a video game.",
        "a sketch of a {}.",
        "a doodle of the {}.",
        "a origami {}.",
        "a low resolution photo of a {}.",
        "the toy {}.",
        "a rendition of the {}.",
        "a photo of the clean {}.",
        "a photo of a large {}.",
        "a rendition of a {}.",
        "a photo of a nice {}.",
        "a photo of a weird {}.",
        "a blurry photo of a {}.",
        "a cartoon {}.",
        "art of a {}.",
        "a sketch of the {}.",
        "a embroidered {}.",
        "a pixelated photo of a {}.",
        "itap of the {}.",
        "a jpeg photo of the {}.",
        "a good photo of a {}.",
        "a plushie {}.",
        "a photo of the nice {}.",
        "a photo of the small {}.",
        "a photo of the weird {}.",
        "the cartoon {}.",
        "art of the {}.",
        "a drawing of the {}.",
        "a photo of the large {}.",
        "a black and white photo of a {}.",
        "the plushie {}.",
        "a dark photo of a {}.",
        "itap of a {}.",
        "graffiti of the {}.",
        "a toy {}.",
        "itap of my {}.",
        "a photo of a cool {}.",
        "a photo of a small {}.",
        "a tattoo of the {}.",
    ]

    # --- 4. Pre-compute Text Embeddings ---
    class_embeddings = []

    with torch.no_grad():
        for cls in cleaned_classes:
            prompts = [t.format(cls) for t in templates]
            inputs = processor(
                text=prompts, return_tensors="pt", padding=True, truncation=True
            ).to(device)
            txt_feats = model.get_text_features(**inputs)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
            mean_feat = txt_feats.mean(dim=0)
            mean_feat = mean_feat / mean_feat.norm()
            class_embeddings.append(mean_feat)

    class_embeddings = torch.stack(class_embeddings)
    logit_scale = model.logit_scale.exp()

    # --- 5. Inference Loop with TTA ---
    for img_id in json_data:
        img_path = os.path.join(IMAGE_DIR, f"{img_id}.jpg")
        try:
            with Image.open(img_path) as image:
                if image.mode != "RGB":
                    image = image.convert("RGB")

                # Get 4 views: Squish, SquishFlip, Letterbox, LetterboxFlip
                tta_images = get_image_views(image, 224)

                image_inputs = processor(images=tta_images, return_tensors="pt").to(
                    device
                )

                with torch.no_grad():
                    # Extract features
                    image_features = model.get_image_features(**image_inputs)
                    image_features = image_features / image_features.norm(
                        dim=-1, keepdim=True
                    )

                    # Average features (Visual Ensemble)
                    mean_image_feature = image_features.mean(dim=0, keepdim=True)
                    mean_image_feature = mean_image_feature / mean_image_feature.norm(
                        dim=-1, keepdim=True
                    )

                    # Similarity
                    logits = (mean_image_feature @ class_embeddings.T) * logit_scale
                    probs = logits.softmax(dim=1)
                    probs_list = probs[0].tolist()

                # --- 6. Aggressive Thresholding for Precision ---
                # Strategy: Almost exclusively pick Top-1.
                # Only pick Top-2 if it's within 95% confidence of Top-1.
                # This minimizes False Positives drastically.

                top_prob = max(probs_list)

                # Dynamic Threshold: 95% of the top probability
                # Absolute Floor: 25% (must be somewhat confident)
                dynamic_threshold = max(0.25, top_prob * 0.95)

                multi_hot = [1 if p >= dynamic_threshold else 0 for p in probs_list]

                # Fallback: Always predict at least the top class if nothing meets criteria
                if sum(multi_hot) == 0:
                    multi_hot[probs_list.index(top_prob)] = 1

                y_pred_dict[img_id] = multi_hot

                # Cleanup
                del image_inputs, image_features, logits, probs

        except Exception as e:
            print(f"Error {img_id}: {e}")
            continue

    ### END SOLUTION
    return y_pred_dict

### Sử dụng bộ dữ liệu `test.json` để kiểm tra hàm `def run(json,classes)` sau đó precision giữa kết quả dữ đoán và nhãn của các ảnh.


In [34]:
try:
    with open("./VOC_CS431/test.json", "r") as f:
        data_test = json.load(f)
    test_classes = ["dog", "cat", "pottedplant", "chair", "sofa", "sheep", "cow"]

    y_pred_dict = run(data_test, test_classes)
    y_true = []
    y_pred = []
    for img_id, labels in data_test.items():
        gt_vec = [1 if cls in labels.split() else 0 for cls in test_classes]
        y_true.append(gt_vec)
        y_pred.append(y_pred_dict[img_id])
    precision, _, _, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    print("Precision =", precision)

except Exception as e:
    print(f"❌ Lỗi: {type(e).__name__}: {e}")

Precision = 0.8333333333333333
